<a href="https://colab.research.google.com/github/artuurog/3d_reconstruction_colmap/blob/main/colmap_3d_reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D reconstruction from images using COLMAP
## https://colmap.org/

In [1]:
import subprocess, os

# check GPU availability
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU available")

# Install COLMAP
subprocess.run(['apt-get', 'install', '-y', 'colmap'],
               capture_output=True)

# check version
result = subprocess.run(['colmap', '--version'], capture_output=True, text=True)
print("COLMAP:", result.stdout.strip())

# Install Open3D for post-processing
subprocess.run(['pip', 'install', 'open3d', '-q'], capture_output=True)
print("Open3D installed.")

Tue May 12 12:32:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Wrapper for pipeline

In [14]:
import subprocess
import os
from pathlib import Path

class ColmapPipeline:
    def __init__(self, workspace: str, images: str, use_gpu: bool = True):
        self.workspace = Path(workspace)
        self.images    = Path(images)
        self.db        = self.workspace / "database.db"
        self.sparse    = self.workspace / "sparse"
        self.dense     = self.workspace / "dense"
        self.use_gpu   = "1" if use_gpu else "0"

        os.environ["QT_QPA_PLATFORM"] = "offscreen"

        # Create folders
        self.sparse.mkdir(parents=True, exist_ok=True)
        self.dense.mkdir(parents=True, exist_ok=True)

    def _run(self, args: list, step: str):
        """Executes a COLMAP command and prints output."""
        print(f"\n{'='*50}")
        print(f"▶ {step}")
        print(f"{'='*50}")

        env = os.environ.copy()
        env["QT_QPA_PLATFORM"] = "offscreen"

        process = subprocess.Popen(
            args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env
        )
        for line in process.stdout:
            print(line, end='')

        process.wait()
        if process.returncode != 0:
            raise RuntimeError(f"COLMAP failed at step: {step}")
        print(f"✅ {step} completed")

    def feature_extraction(self):
        self._run([
            'colmap', 'feature_extractor',
            '--database_path',             str(self.db),
            '--image_path',                str(self.images),
            '--SiftExtraction.use_gpu',    self.use_gpu,
            '--SiftExtraction.max_image_size', '3200',
            '--SiftExtraction.max_num_features', '8192',
            '--ImageReader.single_camera', '0',
        ], "Feature extraction (SIFT)")

    def feature_matching(self, method='exhaustive'):
        """
        method: 'exhaustive' (< ~300 img) | 'vocab_tree' (large dataset)
        """
        if method == 'vocab_tree':
            # Scarica vocab tree se non presente
            vt_path = '/content/vocab_tree_flickr100K_words32K.bin'
            if not os.path.exists(vt_path):
                print("Download vocab tree...")
                subprocess.run([
                    'wget', '-q', '-O', vt_path,
                    'https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin'
                ])
            args = [
                'colmap', 'vocab_tree_matcher',
                '--database_path',          str(self.db),
                '--SiftMatching.use_gpu',   self.use_gpu,
                '--VocabTreeMatching.vocab_tree_path', vt_path,
            ]
        else:
            args = [
                'colmap', 'exhaustive_matcher',
                '--database_path',        str(self.db),
                '--SiftMatching.use_gpu', self.use_gpu,
            ]
        self._run(args, f"Feature matching ({method})")

    def sparse_reconstruction(self):
        self._run([
            'colmap', 'mapper',
            '--database_path', str(self.db),
            '--image_path',    str(self.images),
            '--output_path',   str(self.sparse),
            '--Mapper.num_threads', '4',
            '--Mapper.init_min_tri_angle', '4',
            '--Mapper.multiple_models', '0',
        ], "SfM — Sparse reconstruction")

    def undistort(self):
        self._run([
            'colmap', 'image_undistorter',
            '--image_path',  str(self.images),
            '--input_path',  str(self.sparse / '0'),
            '--output_path', str(self.dense),
            '--output_type', 'COLMAP',
            '--max_image_size', '2000',
        ], "Undistort images")

    def dense_reconstruction(self, use_gpu_mvs: bool = False):
        if use_gpu_mvs:
            # PatchMatch Stereo (requires CUDA)
            self._run([
                'colmap', 'patch_match_stereo',
                '--workspace_path',   str(self.dense),
                '--workspace_format', 'COLMAP',
                '--PatchMatchStereo.geom_consistency', 'true',
                '--PatchMatchStereo.max_image_size',   '2000',
            ], "MVS — PatchMatch Stereo (GPU)")
        else:
            print("⚠️  No GPU: fallback to depth map fusion")
            self._run([
            'colmap', 'patch_match_stereo',
            '--workspace_path',   str(self.dense),
            '--workspace_format', 'COLMAP',
            '--PatchMatchStereo.geom_consistency',  'false',
            '--PatchMatchStereo.gpu_index',         '-1',    # CPU mode
            '--PatchMatchStereo.num_samples',       '15',    # reduce for speed
            '--PatchMatchStereo.num_iterations',    '3',
            '--PatchMatchStereo.max_image_size',    '1000',  # lower res on CPU
        ], "MVS — PatchMatch Stereo (CPU fallback)")

    def fuse_point_cloud(self):
        output_ply = self.dense / 'fused.ply'
        self._run([
            'colmap', 'stereo_fusion',
            '--workspace_path',   str(self.dense),
            '--workspace_format', 'COLMAP',
            '--input_type',       'geometric',
            '--output_path',      str(output_ply),
            '--StereoFusion.min_num_pixels', '3',
        ], "Stereo fusion — Dense point cloud")
        return output_ply

    def run_full_pipeline(self, matching='exhaustive', dense:bool = False):
        self.feature_extraction()
        self.feature_matching(matching)
        self.sparse_reconstruction()
        if dense:
          self.undistort()
          self.dense_reconstruction(use_gpu_mvs=False)
          return self.fuse_point_cloud()
        else:
          # Export sparse model directly as PLY
          sparse_ply = self.workspace / "sparse_cloud.ply"
          self._run([
              'colmap', 'model_converter',
              '--input_path',  str(self.sparse / '0'),
              '--output_path', str(sparse_ply),
              '--output_type', 'PLY',
          ], "Export sparse point cloud as PLY")
          return sparse_ply

Upload images

In [3]:
from google.colab import drive, files
import zipfile, shutil

# Option A: Google Drive
# drive.mount('/content/drive')
# SOURCE = '/content/drive/MyDrive/mie_immagini/'

# Opzione B: direct upload (uncomment)
uploaded = files.upload()
os.makedirs('/content/images', exist_ok=True)
for fname in uploaded:
    shutil.move(fname, f'/content/images/{fname}')
SOURCE = '/content/images'

# Opzione C: import from zip
# with zipfile.ZipFile('/content/drive/MyDrive/scene.zip') as z:
#     z.extractall('/content/images')
# SOURCE = '/content/images'

Saving IMG_20260512_141609634.jpg to IMG_20260512_141609634.jpg
Saving IMG_20260512_141545462_HDR.jpg to IMG_20260512_141545462_HDR.jpg
Saving IMG_20260512_141546573.jpg to IMG_20260512_141546573.jpg
Saving IMG_20260512_141547965.jpg to IMG_20260512_141547965.jpg
Saving IMG_20260512_141551677.jpg to IMG_20260512_141551677.jpg
Saving IMG_20260512_141553355.jpg to IMG_20260512_141553355.jpg
Saving IMG_20260512_141554742.jpg to IMG_20260512_141554742.jpg
Saving IMG_20260512_141556578.jpg to IMG_20260512_141556578.jpg
Saving IMG_20260512_141557982_HDR.jpg to IMG_20260512_141557982_HDR.jpg
Saving IMG_20260512_141559249_HDR.jpg to IMG_20260512_141559249_HDR.jpg
Saving IMG_20260512_141602282.jpg to IMG_20260512_141602282.jpg
Saving IMG_20260512_141603577.jpg to IMG_20260512_141603577.jpg
Saving IMG_20260512_141604954_HDR.jpg to IMG_20260512_141604954_HDR.jpg
Saving IMG_20260512_141606283.jpg to IMG_20260512_141606283.jpg
Saving IMG_20260512_141608201.jpg to IMG_20260512_141608201.jpg
Saving I

Execute pipeline

In [15]:
pipeline = ColmapPipeline(
    workspace='/content/sfm_project',
    images=SOURCE,
    use_gpu=False # false if no GPU avaiable
)

# For dataset < 300 images use 'exhaustive',
# for large datasets use 'vocab_tree'
fused_ply = pipeline.run_full_pipeline(matching='vocab_tree')
print(f"\n🎉 Point cloud saved in: {fused_ply}")

Output streaming troncato alle ultime 5000 righe.
   5  1.639386e+04    6.16e-05    1.97e+00   5.24e-02   1.00e+00  2.43e+06        1    4.94e-02    3.00e-01
   6  1.639386e+04    1.93e-05    6.81e-01   4.12e-02   9.99e-01  7.29e+06        1    5.31e-02    3.53e-01


Bundle adjustment report
------------------------
    Residuals : 32010
   Parameters : 13834
   Iterations : 7
         Time : 0.35391 [s]
 Initial cost : 0.715877 [px]
   Final cost : 0.715645 [px]
  Termination : Convergence

  => Completed observations: 0
  => Merged observations: 0
  => Filtered observations: 1
  => Changed observations: 0.000062
  => Filtered images: 0

Registering image #54 (12)

  => Image sees 844 / 1534 points

Pose refinement report
----------------------
    Residuals : 1442
   Parameters : 6
   Iterations : 24
         Time : 0.0274761 [s]
 Initial cost : 0.85402 [px]
   Final cost : 0.796729 [px]
  Termination : Convergence

  => Continued observations: 701
  => Added observations: 627

Bundl

Mesh with Poisson reconstruction

In [16]:
import open3d as o3d
import numpy as np

pcd = o3d.io.read_point_cloud(str(fused_ply))
print(f"Pointcloud: {len(pcd.points):,}")

# Estimate orthogonal parameters
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)
pcd.orient_normals_consistent_tangent_plane(100)

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=9, width=0, scale=1.1, linear_fit=False
)

threshold = np.quantile(densities, 0.02)
mask = np.asarray(densities) < threshold
mesh.remove_vertices_by_mask(mask)
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()

mesh_path = '/content/sfm_project/output_mesh.ply'
o3d.io.write_triangle_mesh(mesh_path, mesh)
print(f"✅ Mesh saved: {mesh_path}")
print(f"   Vertices: {len(mesh.vertices):,}  |  Triangles: {len(mesh.triangles):,}")

Pointcloud: 10,962
✅ Mesh saved: /content/sfm_project/output_mesh.ply
   Vertices: 31,926  |  Triangles: 62,927


Download results

In [18]:
from google.colab import files
import zipfile

cloud_path = '/content/sfm_project/sparse_cloud.ply'

with zipfile.ZipFile('/content/results.zip', 'w') as z:
    z.write(str(fused_ply),  'fused.ply')        # dense point cloud
    z.write(mesh_path,       'output_mesh.ply')  # final mesh
    z.write(cloud_path,       'sparse_cloud.ply')  # sparse point cloud

files.download('/content/results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>